<h1 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #1f2937 45%, #334155 100%);
  padding: 16px 20px;
  border-radius: 14px;
  text-align: center;
  box-shadow: 0 12px 30px rgba(0,0,0,.35);
  font-family: Arial, sans-serif;
  margin: 0;
">
Naive Bayes - Analisis de sentimientos
</h1>

<h2 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #2b3136 55%, #4a535a 100%);
  padding: 14px 18px;
  border-radius: 12px;
  text-align: center;
  box-shadow: 0 10px 28px rgba(0,0,0,.18);
  border: 1px solid rgba(255,255,255,.10);
  border-top: 5px solid #38bdf8;
  margin: 0;
  font-size: 20px;
">
Carga de librerias y del dataset
</h2>

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"

possible_paths = [
    Path("./data/raw/playstore_reviews.csv"),
    Path("./playstore_reviews.csv")
]

df = None

for path in possible_paths:
    if path.exists():
        df = pd.read_csv(path)
        print(f"Dataset cargado desde archivo local: {path}")
        break

if df is None:
    df = pd.read_csv(url)
    print("Dataset cargado desde la URL oficial")

print("\nDimensiones del dataset:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

df.head()

Dataset cargado desde la URL oficial

Dimensiones del dataset: (891, 3)

Columnas:
['package_name', 'review', 'polarity']


,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


<h2 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #2b3136 55%, #4a535a 100%);
  padding: 14px 18px;
  border-radius: 12px;
  text-align: center;
  box-shadow: 0 10px 28px rgba(0,0,0,.18);
  border: 1px solid rgba(255,255,255,.10);
  border-top: 5px solid #38bdf8;
  margin: 0;
  font-size: 20px;
">
Preparacion de la variable de texto
</h2>

In [3]:
df = df[["review", "polarity"]].copy()

df["review"] = df["review"].astype(str).str.strip().str.lower()

print("Dimensiones luego de quedarnos con review y polarity:", df.shape)
print("\nValores nulos:")
print(df.isnull().sum())

print("\nDistribucion de la variable objetivo:")
print(df["polarity"].value_counts())

df.head()

Dimensiones luego de quedarnos con review y polarity: (891, 2)

Valores nulos:
review      0
polarity    0
dtype: int64

Distribucion de la variable objetivo:
polarity
0    584
1    307
Name: count, dtype: int64


,review,polarity
0,privacy at least put some option appear offlin...,0
1,"messenger issues ever since the last update, i...",0
2,profile any time my wife or anybody has more t...,0
3,the new features suck for those of us who don'...,0
4,forced reload on uploading pic on replying com...,0


<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
Aqui ya deje solo lo que de verdad importa para el problema. Quite <span style="color:#D11A2A;"><b>package_name</b></span> porque el sentimiento no depende del nombre de la app, sino del contenido de <span style="color:#D11A2A;"><b>review</b></span>. Ademas, normalize el texto con <span style="color:#D11A2A;"><b>strip()</b></span> y <span style="color:#D11A2A;"><b>lower()</b></span> para que palabras iguales no se traten como si fueran distintas solo por espacios o mayusculas.
</div>

<h2 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #2b3136 55%, #4a535a 100%);
  padding: 14px 18px;
  border-radius: 12px;
  text-align: center;
  box-shadow: 0 10px 28px rgba(0,0,0,.18);
  border: 1px solid rgba(255,255,255,.10);
  border-top: 5px solid #38bdf8;
  margin: 0;
  font-size: 20px;
">
Division del dataset y transformacion del texto
</h2>

In [4]:
X = df["review"]
y = df["polarity"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

vec_model = CountVectorizer(stop_words="english")

X_train_vec = vec_model.fit_transform(X_train)
X_test_vec = vec_model.transform(X_test)

print("Shape de X_train original:", X_train.shape)
print("Shape de X_test original:", X_test.shape)
print("Shape de X_train vectorizado:", X_train_vec.shape)
print("Shape de X_test vectorizado:", X_test_vec.shape)

print("\nCantidad de palabras unicas aprendidas:", len(vec_model.vocabulary_))

Shape de X_train original: (712,)
Shape de X_test original: (179,)
Shape de X_train vectorizado: (712, 3272)
Shape de X_test vectorizado: (179, 3272)

Cantidad de palabras unicas aprendidas: 3272


<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
En este paso separo los datos en <span style="color:#D11A2A;"><b>train</b></span> y <span style="color:#D11A2A;"><b>test</b></span> para entrenar y evaluar de forma correcta. Despues uso <span style="color:#D11A2A;"><b>CountVectorizer</b></span>, que convierte cada comentario en numeros contando palabras. Asi el modelo deja de ver texto plano y empieza a ver patrones numericos que si puede aprender.
</div>

<h2 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #2b3136 55%, #4a535a 100%);
  padding: 14px 18px;
  border-radius: 12px;
  text-align: center;
  box-shadow: 0 10px 28px rgba(0,0,0,.18);
  border: 1px solid rgba(255,255,255,.10);
  border-top: 5px solid #38bdf8;
  margin: 0;
  font-size: 20px;
">
Entrenamiento y comparacion de Naive Bayes
</h2>

In [5]:
nb_models = {
    "GaussianNB": GaussianNB(),
    "MultinomialNB": MultinomialNB(),
    "BernoulliNB": BernoulliNB()
}

nb_results = []

for name, model in nb_models.items():
    if name == "GaussianNB":
        X_train_model = X_train_vec.toarray()
        X_test_model = X_test_vec.toarray()
    else:
        X_train_model = X_train_vec
        X_test_model = X_test_vec

    model.fit(X_train_model, y_train)

    y_train_pred = model.predict(X_train_model)
    y_test_pred = model.predict(X_test_model)

    nb_results.append({
        "model": name,
        "accuracy_train": accuracy_score(y_train, y_train_pred),
        "accuracy_test": accuracy_score(y_test, y_test_pred)
    })

nb_results_df = pd.DataFrame(nb_results).sort_values(by="accuracy_test", ascending=False).reset_index(drop=True)
nb_results_df

,model,accuracy_train,accuracy_test
0,MultinomialNB,0.955056,0.854749
1,GaussianNB,0.981742,0.815642
2,BernoulliNB,0.924157,0.782123


<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
El mejor modelo fue <span style="color:#D11A2A;"><b>MultinomialNB</b></span>, porque logro el mayor <span style="color:#D11A2A;"><b>accuracy_test = 0.854749</b></span>. Esto confirma que fue la eleccion correcta para este problema, ya que trabaja muy bien con <span style="color:#D11A2A;"><b>conteos de palabras</b></span>, que es exactamente lo que genera <span style="color:#D11A2A;"><b>CountVectorizer</b></span>. Aunque <span style="color:#D11A2A;"><b>GaussianNB</b></span> saco un mejor resultado en entrenamiento, en test quedo por debajo, asi que no generalizo tan bien. <span style="color:#D11A2A;"><b>BernoulliNB</b></span> fue el mas flojo de los tres, por lo que en este dataset la simple presencia o ausencia de palabras no fue tan util como el recuento de frecuencias.
</div>

<h2 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #2b3136 55%, #4a535a 100%);
  padding: 14px 18px;
  border-radius: 12px;
  text-align: center;
  box-shadow: 0 10px 28px rgba(0,0,0,.18);
  border: 1px solid rgba(255,255,255,.10);
  border-top: 5px solid #38bdf8;
  margin: 0;
  font-size: 20px;
">
Evaluacion del mejor modelo
</h2>

In [6]:
best_nb_model = MultinomialNB()
best_nb_model.fit(X_train_vec, y_train)

y_train_pred_best = best_nb_model.predict(X_train_vec)
y_test_pred_best = best_nb_model.predict(X_test_vec)

print("Accuracy train:", accuracy_score(y_train, y_train_pred_best))
print("Accuracy test:", accuracy_score(y_test, y_test_pred_best))

print("\nClassification report en test:\n")
print(classification_report(y_test, y_test_pred_best))

print("Matriz de confusion:\n")
print(confusion_matrix(y_test, y_test_pred_best))

Accuracy train: 0.9550561797752809
Accuracy test: 0.8547486033519553

Classification report en test:

              precision    recall  f1-score   support

           0       0.84      0.96      0.90       117
           1       0.89      0.66      0.76        62

    accuracy                           0.85       179
   macro avg       0.87      0.81      0.83       179
weighted avg       0.86      0.85      0.85       179

Matriz de confusion:

[[112   5]
 [ 21  41]]


<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
El modelo <span style="color:#D11A2A;"><b>MultinomialNB</b></span> dio un resultado bueno en general, con <span style="color:#D11A2A;"><b>accuracy_test = 0.8547</b></span>. Donde mejor trabaja es en los comentarios <span style="color:#D11A2A;"><b>negativos (clase 0)</b></span>, ya que detecto <span style="color:#D11A2A;"><b>112 de 117</b></span>, con un <span style="color:#D11A2A;"><b>recall de 0.96</b></span>. En cambio, con los comentarios <span style="color:#D11A2A;"><b>positivos (clase 1)</b></span> baja mas, porque solo detecto <span style="color:#D11A2A;"><b>41 de 62</b></span>, con un <span style="color:#D11A2A;"><b>recall de 0.66</b></span>. En resumen, el modelo funciona bien, pero esta mas fuerte para reconocer reseñas negativas que positivas.
</div>

<h2 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #2b3136 55%, #4a535a 100%);
  padding: 14px 18px;
  border-radius: 12px;
  text-align: center;
  box-shadow: 0 10px 28px rgba(0,0,0,.18);
  border: 1px solid rgba(255,255,255,.10);
  border-top: 5px solid #38bdf8;
  margin: 0;
  font-size: 20px;
">
Comparacion con Random Forest
</h2>

<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
En este bloque comparo el mejor <span style="color:#D11A2A;"><b>Naive Bayes</b></span> contra <span style="color:#D11A2A;"><b>RandomForest</b></span>. La idea es comprobar si un modelo mas complejo realmente mejora el resultado o si, por el contrario, <span style="color:#D11A2A;"><b>MultinomialNB</b></span> ya resuelve mejor este problema de texto.
</div>

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=42
)

rf_model.fit(X_train_vec, y_train)

y_train_pred_rf = rf_model.predict(X_train_vec)
y_test_pred_rf = rf_model.predict(X_test_vec)

comparison_df = pd.DataFrame({
    "model": ["MultinomialNB", "RandomForest"],
    "accuracy_train": [
        accuracy_score(y_train, y_train_pred_best),
        accuracy_score(y_train, y_train_pred_rf)
    ],
    "accuracy_test": [
        accuracy_score(y_test, y_test_pred_best),
        accuracy_score(y_test, y_test_pred_rf)
    ]})

comparison_df

,model,accuracy_train,accuracy_test
0,MultinomialNB,0.955056,0.854749
1,RandomForest,0.859551,0.765363


<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
<span style="color:#D11A2A;"><b>RandomForest</b></span> no logro mejorar a <span style="color:#D11A2A;"><b>MultinomialNB</b></span>. De hecho, quedo bastante por debajo en test, con <span style="color:#D11A2A;"><b>0.7654</b></span> frente a <span style="color:#D11A2A;"><b>0.8547</b></span>. Esto confirma que para este problema de texto basado en <span style="color:#D11A2A;"><b>conteos de palabras</b></span>, el modelo mas adecuado sigue siendo <span style="color:#D11A2A;"><b>MultinomialNB</b></span>. Aqui un modelo mas complejo no significo un mejor resultado.
</div>

<h2 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #2b3136 55%, #4a535a 100%);
  padding: 14px 18px;
  border-radius: 12px;
  text-align: center;
  box-shadow: 0 10px 28px rgba(0,0,0,.18);
  border: 1px solid rgba(255,255,255,.10);
  border-top: 5px solid #38bdf8;
  margin: 0;
  font-size: 20px;
">
Prueba con una alternativa mas fuerte para texto
</h2>

<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
Aqui pruebo <span style="color:#D11A2A;"><b>LogisticRegression</b></span> porque suele ser una de las mejores alternativas en problemas de clasificacion de texto. La idea es ver si puede superar a <span style="color:#D11A2A;"><b>MultinomialNB</b></span> manteniendo una buena generalizacion sobre datos no vistos.
</div>

In [8]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)

lr_model.fit(X_train_vec, y_train)

y_train_pred_lr = lr_model.predict(X_train_vec)
y_test_pred_lr = lr_model.predict(X_test_vec)

alternatives_df = pd.DataFrame({
    "model": ["MultinomialNB", "RandomForest", "LogisticRegression"],
    "accuracy_train": [
        accuracy_score(y_train, y_train_pred_best),
        accuracy_score(y_train, y_train_pred_rf),
        accuracy_score(y_train, y_train_pred_lr)
    ],
    "accuracy_test": [
        accuracy_score(y_test, y_test_pred_best),
        accuracy_score(y_test, y_test_pred_rf),
        accuracy_score(y_test, y_test_pred_lr)
    ]
})

alternatives_df

,model,accuracy_train,accuracy_test
0,MultinomialNB,0.955056,0.854749
1,RandomForest,0.859551,0.765363
2,LogisticRegression,0.998596,0.832402


<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
<span style="color:#D11A2A;"><b>LogisticRegression</b></span> tampoco logro superar a <span style="color:#D11A2A;"><b>MultinomialNB</b></span>. Aunque en entrenamiento saco un valor casi perfecto de <span style="color:#D11A2A;"><b>0.9986</b></span>, en test bajo a <span style="color:#D11A2A;"><b>0.8324</b></span>, quedando por debajo del <span style="color:#D11A2A;"><b>0.8547</b></span> de <span style="color:#D11A2A;"><b>MultinomialNB</b></span>. Esto indica que, en este dataset, el modelo que mejor generalizo fue <span style="color:#D11A2A;"><b>MultinomialNB</b></span>. Ademas, el resultado de <span style="color:#D11A2A;"><b>LogisticRegression</b></span> sugiere un ajuste muy fuerte al entrenamiento y una generalizacion inferior en comparacion con Naive Bayes.
</div>

<h2 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #2b3136 55%, #4a535a 100%);
  padding: 14px 18px;
  border-radius: 12px;
  text-align: center;
  box-shadow: 0 10px 28px rgba(0,0,0,.18);
  border: 1px solid rgba(255,255,255,.10);
  border-top: 5px solid #38bdf8;
  margin: 0;
  font-size: 20px;
">
Guardado del mejor modelo
</h2>

<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
Aqui guardo tanto el <span style="color:#D11A2A;"><b>modelo</b></span> como el <span style="color:#D11A2A;"><b>vectorizador</b></span>. Esto es importante porque el modelo por si solo no entiende texto plano. Primero necesito transformar el comentario con el mismo <span style="color:#D11A2A;"><b>CountVectorizer</b></span> que use en entrenamiento, y despues pasar ese resultado al modelo para obtener la prediccion.
</div>

In [9]:
models_dir = Path("./models")
models_dir.mkdir(parents=True, exist_ok=True)

with open(models_dir / "multinomial_nb_vectorizer.pkl", "wb") as f:
    pickle.dump(vec_model, f)

with open(models_dir / "multinomial_nb_model.pkl", "wb") as f:
    pickle.dump(best_nb_model, f)

print("Modelo y vectorizador guardados correctamente en la carpeta models")

Modelo y vectorizador guardados correctamente en la carpeta models


<h2 style="
  color: #ffffff;
  background: linear-gradient(135deg, #0b0f12 0%, #2b3136 55%, #4a535a 100%);
  padding: 14px 18px;
  border-radius: 12px;
  text-align: center;
  box-shadow: 0 10px 28px rgba(0,0,0,.18);
  border: 1px solid rgba(255,255,255,.10);
  border-top: 5px solid #38bdf8;
  margin: 0;
  font-size: 20px;
">
Conclusion final
</h2>

<div style="
  background-color: #ececec;
  border-left: 6px solid #22c55e;
  padding: 16px 18px;
  border-radius: 10px;
  box-shadow: 0 4px 14px rgba(0,0,0,.08);
  font-size: 16px;
  line-height: 1.7;
  color: #222;
">
En este proyecto construi un clasificador de sentimientos para reseñas de Google Play usando <span style="color:#D11A2A;"><b>Naive Bayes</b></span>. Primero me quede solo con la variable <span style="color:#D11A2A;"><b>review</b></span>, porque era la unica que realmente aportaba informacion al problema, y transforme el texto en numeros con <span style="color:#D11A2A;"><b>CountVectorizer</b></span>. Luego compare las tres variantes de Naive Bayes y comprobe que <span style="color:#D11A2A;"><b>MultinomialNB</b></span> fue la mejor opcion, con un <span style="color:#D11A2A;"><b>accuracy_test de 0.8547</b></span>.

Al evaluar el modelo en detalle vi que su punto mas fuerte fue detectar comentarios <span style="color:#D11A2A;"><b>negativos</b></span>, donde logro un <span style="color:#D11A2A;"><b>recall de 0.96</b></span>. En los comentarios <span style="color:#D11A2A;"><b>positivos</b></span> tambien funciono bien, pero con menor capacidad de deteccion, ya que su <span style="color:#D11A2A;"><b>recall fue 0.66</b></span>. Esto significa que el modelo generaliza bien, aunque esta mas fuerte para reconocer reseñas negativas que positivas.

Despues probe modelos alternativos para intentar mejorar el resultado. <span style="color:#D11A2A;"><b>RandomForest</b></span> quedo bastante por debajo, con un <span style="color:#D11A2A;"><b>accuracy_test de 0.7654</b></span>, y <span style="color:#D11A2A;"><b>LogisticRegression</b></span> tampoco logro superarlo, ya que alcanzo <span style="color:#D11A2A;"><b>0.8324</b></span>. Con esto confirme que, para este dataset, la mejor combinacion fue <span style="color:#D11A2A;"><b>CountVectorizer + MultinomialNB</b></span>. Finalmente guarde tanto el modelo como el vectorizador para poder reutilizarlos despues sobre comentarios nuevos.
</div>